# So sánh thống kê hai mô hình trên cùng benchmark

Notebook so sánh hai mô hình (gọi tắt là **A** và **B**) trên cùng tập sample đã chấm điểm, dùng:

- **Paired t-test** — chênh lệch trung bình có khác 0 không (giả định gần chuẩn).
- **Wilcoxon signed-rank test** — biến thể không tham số, an toàn khi chênh lệch không chuẩn.
- **Cohen's d** — kích thước hiệu ứng, cho biết khác biệt có *lớn về mặt thực tế* hay chỉ "đáng kể về thống kê".

Các chỉ số dùng để so sánh: `faithfulness_combined`, `expansion_combined`, `vibe_combined`, và `overall` (trung bình ba chỉ số trên).

**Cách dùng:** chỉ cần đổi `FILE_A` / `FILE_B` ở cell đầu — `LABEL_A` / `LABEL_B` sẽ tự suy ra từ tên file. Quy ước `diff = A − B`, nên `mean_diff > 0` ⇒ A cao hơn; `< 0` ⇒ B cao hơn.

In [5]:
import re
from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy import stats

# A = baseline thương mại, B = mô hình của ta. Đổi đường dẫn → label tự cập nhật.
FILE_A = Path("../results/evaluations/md-gpt-5.4-mini_21-05-2026_14-31_eval.json")
FILE_B = Path("../results/evaluations/qwen3.5-4b-facebook-content_nothinking_22-05-2026_06-05_eval.json")

def label_from_path(path: Path) -> str:
    """Suy label gọn từ tên file dạng `<model>_<DD-MM-YYYY>_<HH-MM>_eval.json`."""
    stem = path.stem
    stem = re.sub(r"_eval$", "", stem)
    stem = re.sub(r"_\d{2}-\d{2}-\d{4}_\d{2}-\d{2}$", "", stem)
    return stem

LABEL_A = label_from_path(FILE_A)
LABEL_B = label_from_path(FILE_B)

print(f"A = {LABEL_A}  ({FILE_A.name})")
print(f"B = {LABEL_B}  ({FILE_B.name})")

A = md-gpt-5.4-mini  (md-gpt-5.4-mini_21-05-2026_14-31_eval.json)
B = qwen3.5-4b-facebook-content_nothinking  (qwen3.5-4b-facebook-content_nothinking_22-05-2026_06-05_eval.json)


In [6]:
def load_eval(path: Path) -> pd.DataFrame:
    with Path(path).open("r", encoding="utf-8") as f:
        return pd.DataFrame(json.load(f))

model_a = load_eval(FILE_A)
model_b = load_eval(FILE_B)

metrics = ["faithfulness_combined", "expansion_combined", "vibe_combined"]
for df in (model_a, model_b):
    df["overall"] = df[metrics].astype(float).mean(axis=1)
analysis_metrics = metrics + ["overall"]

assert set(model_a["id"]) == set(model_b["id"]), "Hai file phải có cùng tập id để ghép cặp."
print(f"{LABEL_A}: {len(model_a)} sample")
print(f"{LABEL_B}: {len(model_b)} sample")
print(f"Trùng id (paired-ready): True")

md-gpt-5.4-mini: 100 sample
qwen3.5-4b-facebook-content_nothinking: 100 sample
Trùng id (paired-ready): True


## 1. Thống kê mô tả

Trung bình, độ lệch chuẩn, min, max của từng metric cho hai mô hình. Dùng để cảm nhận "khoảng cách trung bình" và "độ ổn định" trước khi đi vào kiểm định.

In [7]:
def describe(df, label):
    return pd.DataFrame([{
        "model": label,
        "metric": m,
        "mean": df[m].mean(),
        "std": df[m].std(ddof=1),
        "min": df[m].min(),
        "max": df[m].max(),
    } for m in analysis_metrics])

summary = pd.concat(
    [describe(model_a, LABEL_A), describe(model_b, LABEL_B)],
    ignore_index=True,
)
display(summary.round(4))

,model,metric,mean,std,min,max
0,md-gpt-5.4-mini,faithfulness_combined,0.6588,0.0987,0.2333,0.8545
1,md-gpt-5.4-mini,expansion_combined,0.8170,0.0692,0.4777,0.9100
2,md-gpt-5.4-mini,vibe_combined,0.6451,0.0808,0.4462,0.8125
3,md-gpt-5.4-mini,overall,0.7069,0.0550,0.3858,0.8071
4,qwen3.5-4b-facebook-content_nothinking,faithfulness_combined,0.6366,0.0955,0.1500,0.9000
5,qwen3.5-4b-facebook-content_nothinking,expansion_combined,0.7766,0.0709,0.5150,0.9100
6,qwen3.5-4b-facebook-content_nothinking,vibe_combined,0.6482,0.0856,0.3508,0.8831
7,qwen3.5-4b-facebook-content_nothinking,overall,0.6871,0.0521,0.4667,0.8304


## 2. So sánh từng sample (paired)

Ghép cặp theo `id`, tính `diff = score_A − score_B` cho mỗi sample. Bảng dưới cho thấy chênh lệch trung bình đến từ "thắng đều khắp" hay "vài cú thắng lớn".

In [8]:
merged = pd.merge(
    model_a[["id"] + analysis_metrics],
    model_b[["id"] + analysis_metrics],
    on="id", suffixes=("_A", "_B"), validate="one_to_one",
)
for m in analysis_metrics:
    merged[f"{m}_diff"] = merged[f"{m}_A"] - merged[f"{m}_B"]

wins = pd.DataFrame([{
    "metric": m,
    f"{LABEL_A} wins": int((merged[f"{m}_diff"] > 0).sum()),
    f"{LABEL_B} wins": int((merged[f"{m}_diff"] < 0).sum()),
    "tie": int((merged[f"{m}_diff"] == 0).sum()),
    "mean_diff (A-B)": merged[f"{m}_diff"].mean(),
    "median_diff": merged[f"{m}_diff"].median(),
    "std_diff": merged[f"{m}_diff"].std(ddof=1),
} for m in analysis_metrics])
display(wins.round(4))

,metric,md-gpt-5.4-mini wins,qwen3.5-4b-facebook-content_nothinking wins,tie,mean_diff (A-B),median_diff,std_diff
0,faithfulness_combined,51,31,18,0.0221,0.0125,0.0942
1,expansion_combined,68,27,5,0.0404,0.0237,0.0820
2,vibe_combined,44,43,13,-0.0031,0.0000,0.0989
3,overall,67,33,0,0.0198,0.0200,0.0539


In [9]:
# Một ví dụ cặp bị tie để thấy trực quan: lấy sample đầu tiên có diff == 0
# ở bất kỳ metric nào, hiển thị scores của cả hai mô hình.
tie_metric = next(
    (m for m in analysis_metrics if (merged[f"{m}_diff"] == 0).any()),
    None,
)

if tie_metric is None:
    print("Không có cặp nào tie ở bất kỳ metric nào.")
else:
    sample = merged[merged[f"{tie_metric}_diff"] == 0].iloc[0]
    print(f"Ví dụ tie: sample id={int(sample['id'])}, tie ở metric `{tie_metric}`\n")
    example = pd.DataFrame({
        "metric": analysis_metrics,
        LABEL_A: [sample[f"{m}_A"] for m in analysis_metrics],
        LABEL_B: [sample[f"{m}_B"] for m in analysis_metrics],
        "diff": [sample[f"{m}_diff"] for m in analysis_metrics],
    })
    display(example.round(6))

Ví dụ tie: sample id=9, tie ở metric `faithfulness_combined`



,metric,md-gpt-5.4-mini,qwen3.5-4b-facebook-content_nothinking,diff
0,faithfulness_combined,0.677778,0.677778,0.00000
1,expansion_combined,0.886330,0.773000,0.11333
2,vibe_combined,0.581875,0.631875,-0.05000
3,overall,0.715328,0.694218,0.02111


## 3. Kiểm định thống kê: paired t-test + Wilcoxon + Cohen's d

Cả ba được gộp trong **một bảng** để đọc song song:

| Phép kiểm định | Trả lời câu hỏi | Cách đọc |
|---|---|---|
| **Paired t-test** | Trung bình chênh lệch có khác 0 không? | `t_p < 0.05` ⇒ khác biệt có ý nghĩa thống kê |
| **Wilcoxon signed-rank** | Như trên, không cần giả định phân phối chuẩn | `w_p < 0.05` ⇒ khác biệt có ý nghĩa thống kê |
| **Cohen's d** | Khác biệt có lớn về *mặt thực tế* không? | \|d\| < 0.2 nhỏ · < 0.5 vừa · < 0.8 lớn · ≥ 0.8 rất lớn |

**Dấu** của `mean_diff`, `t_stat`, `cohen_d`:
- Dương ⇒ A cao hơn B.
- Âm ⇒ B cao hơn A.

In [ ]:
def effect_label(d: float) -> str:
    a = abs(d)
    if a < 0.2: return "small"
    if a < 0.5: return "medium"
    if a < 0.8: return "large"
    return "very large"

rows = []
for m in analysis_metrics:
    a = merged[f"{m}_A"].astype(float)
    b = merged[f"{m}_B"].astype(float)
    diff = a - b
    t_stat, t_p = stats.ttest_rel(a, b) # T-test
    w_stat, w_p = stats.wilcoxon(diff) # Wilcoxon signed-rank test
    d = diff.mean() / diff.std(ddof=1) # Cohen's d cho paired samples
    rows.append({
        "metric": m,
        "mean_diff (A-B)": diff.mean(),
        "t_stat": t_stat,
        "t_p": t_p,
        "t_sig": t_p < 0.05,
        "wilcoxon_stat": w_stat,
        "w_p": w_p,
        "w_sig": w_p < 0.05,
        "cohen_d": d,
        "effect": effect_label(d),
    })

tests = pd.DataFrame(rows)
display(tests.round(6))

,metric,mean_diff (A-B),t_stat,t_p,t_sig,wilcoxon_stat,w_p,w_sig,cohen_d,effect
0,faithfulness_combined,0.022131,2.349480,0.020785,True,1100.0,0.005372,True,0.234948,medium
1,expansion_combined,0.040442,4.932806,0.000003,True,995.5,0.000002,True,0.493281,medium
2,vibe_combined,-0.003126,-0.316162,0.752545,False,1910.0,0.986492,False,-0.031616,small
3,overall,0.019816,3.674727,0.000387,True,1491.0,0.000378,True,0.367473,medium


## 4. Kết luận từng metric

Diễn giải tự động theo cả ba phép kiểm định: với mỗi metric, cho biết có chênh lệch ý nghĩa thống kê không, bên nào cao hơn, và mức độ hiệu ứng (small / medium / large / very large).

In [11]:
def verdict(row) -> str:
    mean_diff = row["mean_diff (A-B)"]
    t_p, w_p, d = row["t_p"], row["w_p"], row["cohen_d"]
    sig = (t_p < 0.05) and (w_p < 0.05)
    winner = LABEL_A if mean_diff > 0 else LABEL_B
    if not sig:
        return (f"không có khác biệt rõ ràng "
                f"(t_p={t_p:.4f}, w_p={w_p:.4f}, |d|={abs(d):.3f} — {effect_label(d)})")
    return (f"{winner} cao hơn có ý nghĩa thống kê "
            f"(t_p={t_p:.4f}, w_p={w_p:.4f}, d={d:+.3f} — {effect_label(d)})")

print(f"So sánh {LABEL_A} (A) vs {LABEL_B} (B) trên {len(merged)} sample:\n")
for _, row in tests.iterrows():
    print(f"- {row['metric']:25s}: {verdict(row)}")

So sánh md-gpt-5.4-mini (A) vs qwen3.5-4b-facebook-content_nothinking (B) trên 100 sample:

- faithfulness_combined    : md-gpt-5.4-mini cao hơn có ý nghĩa thống kê (t_p=0.0208, w_p=0.0054, d=+0.235 — medium)
- expansion_combined       : md-gpt-5.4-mini cao hơn có ý nghĩa thống kê (t_p=0.0000, w_p=0.0000, d=+0.493 — medium)
- vibe_combined            : không có khác biệt rõ ràng (t_p=0.7525, w_p=0.9865, |d|=0.032 — small)
- overall                  : md-gpt-5.4-mini cao hơn có ý nghĩa thống kê (t_p=0.0004, w_p=0.0004, d=+0.367 — medium)
